In [3]:
# ==========================================
# 1. تثبيت المكتبات المطلوبة
# ==========================================
!pip install -q ipywidgets transformers torch huggingface_hub scikit-learn

import ipywidgets as widgets
from IPython.display import display, clear_output
import torch
import torch.nn as nn
import numpy as np
import time
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download

# تحديد الجهاز (GPU إذا توفر)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ البيئة جاهزة وتعمل على: {device}")

# ==========================================
# 2. تحميل أدوات الذكاء الاصطناعي (CodeBERT)
# ==========================================
print("⏳ جاري تحميل محرك استخراج المتجهات (CodeBERT)...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
base_model = AutoModel.from_pretrained("microsoft/codebert-base").to(device)

def get_embeddings(code):
    inputs = tokenizer(code, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = base_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

# ==========================================
# 3. تعريف معمارية النموذج الهجين (Triple Fusion)
# ==========================================
class TripleFusionSentinel(nn.Module):
    def __init__(self, input_dim=768, expert_dim=4, hidden_dim=256, num_classes=4):
        super().__init__()
        self.static_net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(0.5))
        self.dynamic_net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(0.5))
        self.expert_net = nn.Sequential(nn.Linear(expert_dim, 64), nn.ReLU(), nn.Dropout(0.3))
        
        fusion_dim = (hidden_dim * 2) + 64
        self.attention = nn.Sequential(nn.Linear(fusion_dim, fusion_dim // 2), nn.Tanh(), nn.Linear(fusion_dim // 2, fusion_dim), nn.Sigmoid())
        self.classifier = nn.Sequential(nn.Linear(fusion_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.6), nn.Linear(128, num_classes))

    def forward(self, static_emb, dynamic_emb, expert_feat):
        s_feat = self.static_net(static_emb)
        d_feat = self.dynamic_net(dynamic_emb)
        e_feat = self.expert_net(expert_feat)
        combined = torch.cat((s_feat, d_feat, e_feat), dim=1)
        fused = combined * self.attention(combined)
        return self.classifier(fused)

# ==========================================
# 4. تحميل أوزان النموذج من Hugging Face
# ==========================================
print("⏳ جاري سحب أوزان النموذج المدرب...")
REPO_ID = "maherghanem86/Web3-Smart-Contract-Auditor"
FILENAME = "hybrid_fusion_results/best_fusion_model.pth"

try:
    model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
    model = TripleFusionSentinel().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print("✅ تم تحميل النموذج وجاهز للعمل!")
except Exception as e:
    print(f"❌ خطأ في تحميل النموذج: {e}")

# ==========================================
# 5. منطق التنبؤ والواجهة (UI Logic)
# ==========================================
labels_list = ["High", "Low", "Medium", "None"]
labels_map = {0: "High 🔴", 1: "Low 🟡", 2: "Medium 🟠", 3: "None 🟢"}

def predict_contract(code):
    static_emb = get_embeddings(code)
    dynamic_emb = static_emb 
    expert_feat = np.array([np.mean(static_emb), np.std(static_emb), np.max(dynamic_emb), np.linalg.norm(static_emb)])

    t_static = torch.tensor(static_emb, dtype=torch.float32).unsqueeze(0).to(device)
    t_dynamic = torch.tensor(dynamic_emb, dtype=torch.float32).unsqueeze(0).to(device)
    t_expert = torch.tensor(expert_feat, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(t_static, t_dynamic, t_expert)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0).cpu().numpy()

    return labels_list[np.argmax(probabilities)], probabilities

# --- عناصر الواجهة ---
title = widgets.HTML("<h2 style='color:#2E86C1; text-align:center;'>🛡️ Smart Contract Security Auditor</h2>")
subtitle = widgets.HTML("<p style='text-align:center;'><b>تحليل أمان العقود الذكية - إطار عمل Triple Fusion</b></p>")
code_input = widgets.Textarea(placeholder='انسخ كود Solidity هنا...', layout=widgets.Layout(width='100%', height='250px'))
analyze_btn = widgets.Button(description="🔍 بدء التحليل الذكي", button_style='success', layout=widgets.Layout(width='100%', margin='10px 0'))
output_area = widgets.Output()

def on_click(b):
    with output_area:
        clear_output()
        if not code_input.value.strip():
            display(widgets.HTML("<b style='color:red'>⚠️ يرجى إدخال الكود!</b>"))
            return
        
        # شريط انتظار
        progress = widgets.IntProgress(value=0, min=0, max=100, description='Processing:', bar_style='info', layout=widgets.Layout(width='100%'))
        display(progress)
        for i in range(1, 101, 20): 
            time.sleep(0.1)
            progress.value = i
        
        label, probs = predict_contract(code_input.value)
        clear_output()
        
        # الألوان
        colors = {"None": "#28B463", "Low": "#F1C40F", "Medium": "#E67E22", "High": "#C0392B"}
        
        display(widgets.HTML(f"<div style='background-color:{colors[label]}; padding:10px; border-radius:5px; color:white; text-align:center;'>"
                             f"<h3>النتيجة النهائية: {label}</h3></div>"))
        
        display(widgets.HTML("<h4 style='margin-top:15px;'>📊 تفاصيل احتمالات الفئات:</h4>"))
        
        # عرض الأشرطة مع الأرقام بجانبها
        for i, p in enumerate(probs):
            percent = p * 100
            bar = widgets.FloatProgress(
                value=float(p), min=0, max=1.0, 
                description=labels_map[i], 
                bar_style='info', 
                layout=widgets.Layout(width='75%')
            )
            # إضافة النص الذي يحتوي على النسبة المئوية
            percent_text = widgets.HTML(f"<b style='margin-left:10px;'>{percent:.2f}%</b>")
            
            # دمج الشرايط مع النص في صندوق أفقي (HBox)
            row = widgets.HBox([bar, percent_text])
            display(row)

analyze_btn.on_click(on_click)
display(widgets.VBox([title, subtitle, widgets.HTML("<hr>"), code_input, analyze_btn, output_area], 
                     layout=widgets.Layout(padding='20px', border='1px solid #ddd', border_radius='10px')))

⚙️ البيئة جاهزة وتعمل على: cuda
⏳ جاري تحميل محرك استخراج المتجهات (CodeBERT)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

⏳ جاري سحب أوزان النموذج المدرب...
✅ تم تحميل النموذج وجاهز للعمل!
